# 03 Granite briefings

Load Granite once, measure extraction speed, then generate and inspect grounded briefings for the demo snapshot. Until the Granite file and demo outputs arrive, Run All uses the offline fake LLM and checked-in fixture data. Fake measurements are labeled below.

In [ ]:
# Bootstrap: make the repo importable whether the notebook runs in IBM's container or ours
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from surgedesk.envcheck import run
ENV = run()

In [ ]:
# Keep Granite loading in this cell so verbose llama.cpp backend logs can be captured.
import time
from surgedesk import config, llm as llm_module
from surgedesk.contracts import BRIEF_FIELDS

# Resolve the source before opening the model so its call log lands with the output.
demo_dir = config.event_dir(config.DEMO_EVENT)
required = [config.REPORTS_FILE, config.TAGGED_FILE,
            config.EMBEDDINGS_FILE, config.INCIDENTS_FILE]
demo_ready = all((demo_dir / name).is_file() for name in required)
event = config.DEMO_EVENT if demo_ready else config.FIXTURE_EVENT
src_dir = demo_dir if demo_ready else config.FIXTURES / 'outputs' / event
out_dir = config.event_dir(event)

class OfflineFakeLLM:
    """A deterministic, offline stand-in until the model file arrives."""
    model_file = 'fake'
    load_s = 0.0

    @staticmethod
    def _timing(start):
        return {'seconds': time.perf_counter() - start, 'prompt_tokens': 0,
                'completion_tokens': 0, 'ok': True, 'attempts': 1}

    def chat_json(self, system, user, schema, max_tokens):
        start = time.perf_counter()
        fields = {name: {'value': None, 'evidence': None, 'confidence': 0.0}
                  for name in BRIEF_FIELDS}
        return fields, self._timing(start)

    def chat_text(self, system, user, max_tokens):
        start = time.perf_counter()
        return 'No grounded facts available.', self._timing(start)

model_path = config.GRANITE_PATH if config.GRANITE_PATH.is_absolute() else ROOT / config.GRANITE_PATH
if model_path.is_file():
    import hashlib
    model_digest = hashlib.sha256()
    with model_path.open('rb') as model_stream:
        for chunk in iter(lambda: model_stream.read(8 * 1024 * 1024), b''):
            model_digest.update(chunk)
    print(f'Model SHA-256: {model_digest.hexdigest()}')
    active_llm = llm_module.get_llm(
        model_path=model_path, verbose=True, log_path=out_dir / config.LLM_LOG_FILE
    )
    fake_mode = False
    model_bytes = model_path.stat().st_size
    print(f'Granite load: {active_llm.load_s:.3f} s')
    print(f'Model file: {active_llm.model_file} ({model_bytes:,} bytes)')
else:
    active_llm = OfflineFakeLLM()
    fake_mode = True
    print('Granite file is absent; using the offline fake LLM. Model load and speed are not measured.')

In [ ]:
# Five one-report extractions. Fixture data stands in until upstream demo files exist.
import pandas as pd
from surgedesk import brief
from surgedesk.contracts import Report
from surgedesk.fileio import read_jsonl

print(f'Benchmark source: {event}' + (' (fixture)' if not demo_ready else ''))
reports = read_jsonl(src_dir / config.REPORTS_FILE, Report)
sample = reports[:5]
assert len(sample) == 5, 'Speed test requires five reports'
bench_rows = []
for report in sample:
    _, timing = brief.extract_fields(active_llm, [report])
    bench_rows.append({'report_id': report.report_id,
                       'seconds': timing['seconds'],
                       'completion_tokens': timing['completion_tokens']})
print(pd.DataFrame(bench_rows).to_string(index=False))
seconds = sum(row['seconds'] for row in bench_rows)
tokens = sum(row['completion_tokens'] for row in bench_rows)
if fake_mode:
    print(f'Mean seconds per call: {seconds / len(bench_rows):.6f} (fake LLM; not model speed)')
    print('Completion tokens/sec: not measured (fake LLM)')
else:
    print(f'Mean seconds per call: {seconds / len(bench_rows):.3f}')
    print(f'Completion tokens/sec: {tokens / seconds:.3f}' if seconds else 'Completion tokens/sec: unavailable')
try:
    import psutil
except ImportError:
    print('Process RSS unavailable: psutil is not installed in this local kernel.')
else:
    rss_bytes = psutil.Process().memory_info().rss
    print(f'Process RSS: {rss_bytes / (1024 ** 3):.3f} GiB')

In [ ]:
# The normal demo path uses Agent 2's snapshot choice and fairness adjustment.
from surgedesk.contracts import Incident
incidents = read_jsonl(src_dir / config.INCIDENTS_FILE, Incident)
if demo_ready:
    from surgedesk import fairness, rank
    snap = rank.pick_demo_snapshot(incidents)
    at_snap = fairness.apply([i for i in incidents if i.snapshot_time == snap], lam=0.0)
else:
    # Checked-in fixtures have final-snapshot ranks, and can run before Agent 2 lands.
    snap = max(i.snapshot_time for i in incidents)
    at_snap = sorted([i for i in incidents if i.snapshot_time == snap],
                     key=lambda i: i.rank_adj)
    print('Using the fixture final snapshot and its precomputed ranks.')
briefings = brief.brief_incidents(
    event, at_snap, llm=active_llm, src_dir=src_dir, out_dir=out_dir
)
print(f'Snapshot: {snap}; briefings: {len(briefings)}; output: {out_dir / config.BRIEFINGS_FILE}')

In [ ]:
# Display only masked text, including model output and fixture places.
from surgedesk.textutil import mask_privacy
incident_by_id = {item.incident_id: item for item in at_snap}
for result in briefings:
    incident = incident_by_id[result.incident_id]
    print(f'\nRank {incident.rank_adj}: {result.incident_id}')
    print('Place:', mask_privacy(incident.place or 'unknown'))
    for name in BRIEF_FIELDS:
        field = result.fields[name]
        quote = mask_privacy(field['evidence']) if field['evidence'] else 'unknown'
        print(f'  {name}: {quote} (grounded={field["grounded"]})')
    for sentence in result.sentences:
        label = 'accepted' if sentence['accepted'] else f'rejected: {sentence["reject_reason"]}'
        print(f'  {label}: {mask_privacy(sentence["text"])}')
    print('Briefing:', mask_privacy(result.briefing))
    print('Used fallback:', result.used_fallback)

In [ ]:
# These counts are computed from the returned briefing records.
field_total = len(briefings) * len(BRIEF_FIELDS)
field_grounded = sum(result.fields[name]['grounded']
                     for result in briefings for name in BRIEF_FIELDS)
sentence_total = sum(len(result.sentences) for result in briefings)
sentence_accepted = sum(sentence['accepted']
                        for result in briefings for sentence in result.sentences)
fallback_count = sum(result.used_fallback for result in briefings)
mean_seconds = (sum(result.timing['extract_s'] + result.timing['brief_s']
                    for result in briefings) / len(briefings)) if briefings else 0.0
print(f'Fields grounded: {field_grounded}/{field_total}')
print(f'Sentences accepted: {sentence_accepted}/{sentence_total}')
print(f'Fallbacks: {fallback_count}/{len(briefings)}')
if fake_mode:
    print(f'Mean recorded seconds per incident: {mean_seconds:.6f} (fake LLM; not model speed)')
else:
    print(f'Mean recorded seconds per incident: {mean_seconds:.3f}')

### Extraction evaluation

The next cell runs after `data/labels/hand_labels.csv` is available. Fake predictions are stored under stub outputs so Granite can later produce the real evaluation file.


In [ ]:
# Extraction evaluation starts after Person 1 supplies the hand labels.
labels_path = config.LABELS_DIR / 'hand_labels.csv'
demo_reports_path = demo_dir / config.REPORTS_FILE
if labels_path.is_file() and demo_reports_path.is_file():
    labels = pd.read_csv(labels_path, dtype={'report_id': str})
    report_ids = list(dict.fromkeys(labels['report_id'].dropna().tolist()))
    eval_out_dir = config.event_dir(config.DEMO_EVENT, stub=True) if fake_mode else None
    pred_path = brief.extract_for_eval(config.DEMO_EVENT, report_ids, llm=active_llm,
                                       out_dir=eval_out_dir)
    label = 'Offline fake predictions' if fake_mode else 'Extraction predictions'
    print(f'{label}: {pred_path}')
else:
    print('Extraction evaluation runs after labeling and demo reports are available.')

In [ ]:
# Release Granite before another stage or notebook needs RAM.
llm_module.unload()
del active_llm
print('LLM unloaded.')